In [69]:
!pip -q install -U openai gradio pandas python-dotenv

In [68]:
import os, json
import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # loads OPENAI_API_KEY from .env if present

# Set OPENAI_API_KEY in your .env or environment variables
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY. Add it to .env or your environment."

In [70]:
SCHEMAS = {
    "Support Tickets": ["ticket_id","category","priority","message","resolution"],
    "Fintech Transactions": ["tx_id","amount","currency","status","failure_reason"],
}

STYLES = {
    "Normal": "realistic, clean examples",
    "Noisy": "typos, slang, short messages, real-world ambiguity",
    "Edge cases": "rare failure scenarios and tricky cases",
}

In [71]:
SYSTEM_PROMPT = (
    "You are a synthetic dataset generator. "
    "Output ONLY valid JSON (no markdown, no backticks, no explanation). "
    "Return a JSON array of objects with EXACTLY the keys requested."
)

def make_prompt(dataset, n, style):
    fields = SCHEMAS[dataset]
    return f"""
Generate {n} rows of {dataset} as VALID JSON only.

Rules:
- Output must be a JSON array with exactly {n} objects.
- Each object must have EXACTLY these keys: {fields}
- Make it {STYLES[style]}.
- IDs must be unique.
Return ONLY JSON.
""".strip()

In [72]:
def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        # Remove ```json ... ``` or ``` ... ```
        parts = s.split("```")
        if len(parts) >= 2:
            s = parts[1].strip()
        if s.lower().startswith("json"):
            s = s[4:].strip()
    return s

def generate(dataset, n_rows, style, model):
    prompt = make_prompt(dataset, int(n_rows), style)

    # Chat Completions call
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.8,
    )

    raw = resp.choices[0].message.content or ""
    raw = strip_code_fences(raw)

    data = json.loads(raw)
    df = pd.DataFrame(data)[SCHEMAS[dataset]]

    path = "synthetic.csv"  # works locally + colab
    df.to_csv(path, index=False)
    return df, path

In [ ]:
# Pick an OpenAI model you have access to.
# If this model name errors, replace it with one available in your account.
DEFAULT_MODEL = "gpt-4o-mini"

with gr.Blocks() as demo:
    gr.Markdown("# 🧪 Simple Synthetic Data Generator (OpenAI)")

    dataset = gr.Dropdown(list(SCHEMAS.keys()), value="Support Tickets", label="Dataset")
    style = gr.Dropdown(list(STYLES.keys()), value="Normal", label="Style")
    n_rows = gr.Slider(5, 50, value=10, step=1, label="Rows")

    model = gr.Textbox(value=DEFAULT_MODEL, label="OpenAI Model")

    btn = gr.Button("Generate ✅")
    table = gr.Dataframe(interactive=False, label="Preview")
    file = gr.File(label="Download CSV")

    btn.click(generate, inputs=[dataset, n_rows, style, model], outputs=[table, file])

demo.launch(share=False)